# Customer Churn Prediction & Retention Analytics

## Notebook 1: Synthetic SaaS Dataset Generation

### Business Scenario

InsightFlow Analytics Ltd is a B2B SaaS company analysing customer churn.

This notebook generates a realistic synthetic dataset containing:

- Customer profiles
- Monthly product usage behaviour
- Customer support interactions

The dataset will be used for:

- Exploratory Data Analysis
- SQL analysis
- Statistical testing
- Machine learning churn prediction
- Power BI dashboarding

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Random generation
import random

# Dates
from datetime import datetime, timedelta

# Database
import sqlite3

# Display settings
import warnings
warnings.filterwarnings("ignore")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully")

Libraries imported successfully


## Dataset Configuration

The synthetic dataset will simulate:

- 10,000 customers
- 12 months of behaviour history
- Approximately 25,000 support interactions

In [2]:
# Number of customers

NUM_CUSTOMERS = 10000

# Behaviour months

NUM_MONTHS = 12

# Starting customer ID

START_CUSTOMER_ID = 10001


print("Customers:", NUM_CUSTOMERS)
print("Behaviour months:", NUM_MONTHS)

Customers: 10000
Behaviour months: 12


## Creating Customer Master Table

This table contains:

- Customer demographics
- Subscription information
- Revenue
- Tenure
- Churn status

In [3]:
# Customer IDs

customer_ids = range(
    START_CUSTOMER_ID,
    START_CUSTOMER_ID + NUM_CUSTOMERS
)


# Regions

regions = [
    "London",
    "Manchester",
    "Birmingham",
    "Leeds",
    "Bristol",
    "Glasgow"
]


# Age groups

age_groups = [
    "18-25",
    "26-35",
    "36-45",
    "46-60",
    "60+"
]


# Subscription types

subscription_types = [
    "Basic",
    "Professional",
    "Enterprise"
]


customers = pd.DataFrame({

    "customer_id": customer_ids,

    "signup_date": pd.to_datetime(
        np.random.choice(
            pd.date_range(
                "2020-01-01",
                "2025-01-01"
            ),
            NUM_CUSTOMERS
        )
    ),

    "region": np.random.choice(
        regions,
        NUM_CUSTOMERS,
        p=[0.30,0.20,0.15,0.15,0.10,0.10]
    ),

    "age_group": np.random.choice(
        age_groups,
        NUM_CUSTOMERS
    ),

    "subscription_type": np.random.choice(
        subscription_types,
        NUM_CUSTOMERS,
        p=[0.45,0.40,0.15]
    )

})


customers.head()

,customer_id,signup_date,region,age_group,subscription_type
0,10001,2023-01-31,London,60+,Professional
1,10002,2023-12-30,Leeds,18-25,Professional
2,10003,2022-05-10,Bristol,26-35,Basic
3,10004,2023-07-18,London,18-25,Basic
4,10005,2023-02-04,Manchester,60+,Basic


In [4]:
# Revenue mapping

revenue_map = {
    "Basic":29,
    "Professional":79,
    "Enterprise":199
}


customers["monthly_revenue"] = (
    customers["subscription_type"]
    .map(revenue_map)
)


# Calculate tenure

current_date = pd.Timestamp("2025-12-31")


customers["tenure_months"] = (
    (current_date - customers["signup_date"])
    .dt.days
    // 30
)


customers.head()

,customer_id,signup_date,region,age_group,subscription_type,monthly_revenue,tenure_months
0,10001,2023-01-31,London,60+,Professional,79,35
1,10002,2023-12-30,Leeds,18-25,Professional,79,24
2,10003,2022-05-10,Bristol,26-35,Basic,29,44
3,10004,2023-07-18,London,18-25,Basic,29,29
4,10005,2023-02-04,Manchester,60+,Basic,29,35


In [5]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        10000 non-null  int64         
 1   signup_date        10000 non-null  datetime64[ns]
 2   region             10000 non-null  object        
 3   age_group          10000 non-null  object        
 4   subscription_type  10000 non-null  object        
 5   monthly_revenue    10000 non-null  int64         
 6   tenure_months      10000 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 547.0+ KB


In [6]:
customers.head()

,customer_id,signup_date,region,age_group,subscription_type,monthly_revenue,tenure_months
0,10001,2023-01-31,London,60+,Professional,79,35
1,10002,2023-12-30,Leeds,18-25,Professional,79,24
2,10003,2022-05-10,Bristol,26-35,Basic,29,44
3,10004,2023-07-18,London,18-25,Basic,29,29
4,10005,2023-02-04,Manchester,60+,Basic,29,35


In [7]:
customers["subscription_type"].value_counts()

subscription_type
Basic           4560
Professional    4015
Enterprise      1425
Name: count, dtype: int64

## Creating Customer Behaviour Table

This table tracks monthly engagement.

Lower engagement will later contribute to higher churn probability.

In [8]:
# Create 12 monthly periods

months = pd.date_range(
    start="2025-01-01",
    periods=NUM_MONTHS,
    freq="MS"
)


months

DatetimeIndex(['2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01',
               '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01',
               '2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01'],
              dtype='datetime64[ns]', freq='MS')

In [9]:
# Create customer-month combinations

customer_behaviour = pd.MultiIndex.from_product(
    [
        customers["customer_id"],
        months
    ],
    names=[
        "customer_id",
        "month"
    ]
).to_frame(index=False)


customer_behaviour.head()

,customer_id,month
0,10001,2025-01-01
1,10001,2025-02-01
2,10001,2025-03-01
3,10001,2025-04-01
4,10001,2025-05-01


In [10]:
customer_behaviour.shape

(120000, 2)

In [11]:
# Generate login activity

customer_behaviour["monthly_login_count"] = np.random.poisson(
    lam=15,
    size=len(customer_behaviour)
)


# Average session duration

customer_behaviour["average_session_time_minutes"] = np.random.normal(
    loc=25,
    scale=10,
    size=len(customer_behaviour)
)


# Keep values realistic

customer_behaviour["average_session_time_minutes"] = (
    customer_behaviour["average_session_time_minutes"]
    .clip(5,90)
)


# Features used

customer_behaviour["features_used"] = np.random.randint(
    1,
    10,
    size=len(customer_behaviour)
)


# Days since last login

customer_behaviour["last_login_days_ago"] = np.random.randint(
    0,
    60,
    size=len(customer_behaviour)
)


customer_behaviour.head()

,customer_id,month,monthly_login_count,average_session_time_minutes,features_used,last_login_days_ago
0,10001,2025-01-01,13,10.238629,4,0
1,10001,2025-02-01,13,32.184443,8,23
2,10001,2025-03-01,21,24.694519,4,42
3,10001,2025-04-01,16,28.249874,2,36
4,10001,2025-05-01,15,38.528476,9,49


In [12]:
# Generate more realistic SaaS engagement levels

engagement_level = np.random.choice(
    [
        "High",
        "Medium",
        "Low"
    ],
    size=len(customer_behaviour),
    p=[
        0.25,
        0.50,
        0.25
    ]
)


customer_behaviour["monthly_login_count"] = np.where(
    engagement_level == "High",
    np.random.randint(20,40,len(customer_behaviour)),
    np.where(
        engagement_level == "Medium",
        np.random.randint(8,20,len(customer_behaviour)),
        np.random.randint(0,8,len(customer_behaviour))
    )
)

In [13]:
def usage_category(login_count):

    if login_count >= 20:
        return "High"

    elif login_count >= 10:
        return "Medium"

    else:
        return "Low"

In [14]:
customer_behaviour["usage_frequency"] = (
    customer_behaviour["monthly_login_count"]
    .apply(usage_category)
)

In [15]:
customer_behaviour.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 7 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   customer_id                   120000 non-null  int64         
 1   month                         120000 non-null  datetime64[ns]
 2   monthly_login_count           120000 non-null  int64         
 3   average_session_time_minutes  120000 non-null  float64       
 4   features_used                 120000 non-null  int64         
 5   last_login_days_ago           120000 non-null  int64         
 6   usage_frequency               120000 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(4), object(1)
memory usage: 6.4+ MB


In [16]:
customer_behaviour.shape

(120000, 7)

In [17]:
customer_behaviour["usage_frequency"].value_counts()

usage_frequency
Medium    49819
Low       40196
High      29985
Name: count, dtype: int64

# Creating Support Interaction Table

This table records customer service activity:

- Issue category
- Complaint volume
- Response times
- Resolution times
- Satisfaction scores

Poor support experiences will contribute to churn risk.

In [18]:
support_interactions = []

ticket_id = 1

for customer_id in customers["customer_id"]:

    # Probability of having support tickets
    num_tickets = np.random.poisson(1.5)

    for _ in range(num_tickets):

        support_interactions.append({

            "ticket_id": ticket_id,

            "customer_id": customer_id,

            "issue_category": np.random.choice(
                [
                    "Technical Issue",
                    "Billing",
                    "Feature Request",
                    "Account Support"
                ]
            ),

            "complaint_count": np.random.randint(
                1, 4
            ),

            "response_time_hours": np.random.uniform(
                1, 48
            ),

            "resolution_time_hours": np.random.uniform(
                2, 120
            ),

            "customer_satisfaction_score": np.random.randint(
                1, 11
            )

        })

        ticket_id += 1


support_interactions = pd.DataFrame(
    support_interactions
)


support_interactions.head()

,ticket_id,customer_id,issue_category,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score
0,1,10002,Account Support,2,13.014601,56.085322,9
1,2,10003,Technical Issue,1,4.425014,6.192709,7
2,3,10005,Billing,3,22.489549,82.144367,9
3,4,10006,Billing,1,27.525825,2.260798,1
4,5,10006,Feature Request,1,18.788623,80.396478,6


In [19]:
# Realistic support tickets per customer

support_counts = np.random.choice(
    [0,1,2,3,4,5,6,7,8],
    size=NUM_CUSTOMERS,
    p=[
        0.35,
        0.25,
        0.15,
        0.10,
        0.06,
        0.04,
        0.025,
        0.015,
        0.01
    ]
)


support_summary = (

    support_interactions
    .groupby("customer_id")
    .agg({

        "complaint_count": "sum",

        "response_time_hours": "mean",

        "resolution_time_hours": "mean",

        "customer_satisfaction_score": "mean"

    })

    .reset_index()

)


support_summary.head()

,customer_id,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score
0,10002,2,13.014601,56.085322,9.0
1,10003,1,4.425014,6.192709,7.0
2,10005,3,22.489549,82.144367,9.0
3,10006,6,19.815436,48.375721,5.5
4,10008,4,16.224652,27.385024,7.5


In [20]:
support_interactions.shape

(14846, 7)

In [21]:
# Ticket IDs

support_interactions["ticket_id"] = range(
    1,
    len(support_interactions)+1
)


# Issue categories

support_interactions["issue_category"] = np.random.choice(
    [
        "Technical Issue",
        "Billing Question",
        "Feature Request",
        "Performance Problem",
        "Account Issue"
    ],
    len(support_interactions)
)


# Complaint count

support_interactions["complaint_count"] = np.random.randint(
    0,
    4,
    len(support_interactions)
)


# Response time

support_interactions["response_time_hours"] = np.random.gamma(
    shape=2,
    scale=8,
    size=len(support_interactions)
).round(1)


# Resolution time

support_interactions["resolution_time_hours"] = (
    support_interactions["response_time_hours"]
    *
    np.random.uniform(
        1.5,
        4,
        len(support_interactions)
    )
).round(1)


# Satisfaction score

support_interactions["customer_satisfaction_score"] = np.random.randint(
    1,
    11,
    len(support_interactions)
)


support_interactions.head()

,ticket_id,customer_id,issue_category,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score
0,1,10002,Technical Issue,3,33.7,120.7,4
1,2,10003,Technical Issue,3,6.9,15.8,4
2,3,10005,Billing Question,0,5.3,17.9,5
3,4,10006,Technical Issue,3,19.1,48.4,9
4,5,10006,Performance Problem,0,34.9,75.3,4


In [22]:
support_interactions = support_interactions[
[
    "ticket_id",
    "customer_id",
    "issue_category",
    "complaint_count",
    "response_time_hours",
    "resolution_time_hours",
    "customer_satisfaction_score"
]
]


support_interactions.head()

,ticket_id,customer_id,issue_category,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score
0,1,10002,Technical Issue,3,33.7,120.7,4
1,2,10003,Technical Issue,3,6.9,15.8,4
2,3,10005,Billing Question,0,5.3,17.9,5
3,4,10006,Technical Issue,3,19.1,48.4,9
4,5,10006,Performance Problem,0,34.9,75.3,4


In [23]:
support_interactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14846 entries, 0 to 14845
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ticket_id                    14846 non-null  int64  
 1   customer_id                  14846 non-null  int64  
 2   issue_category               14846 non-null  object 
 3   complaint_count              14846 non-null  int64  
 4   response_time_hours          14846 non-null  float64
 5   resolution_time_hours        14846 non-null  float64
 6   customer_satisfaction_score  14846 non-null  int64  
dtypes: float64(2), int64(4), object(1)
memory usage: 812.0+ KB


In [24]:
support_interactions.shape

(14846, 7)

In [25]:
support_interactions.groupby(
    "customer_id"
).size().describe()

count    7717.000000
mean        1.923805
std         1.042836
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         8.000000
dtype: float64

In [26]:
customer_behaviour_summary = (

    customer_behaviour
    .groupby("customer_id")
    .agg({

        "monthly_login_count": "mean",
        "average_session_time_minutes": "mean",
        "features_used": "mean",
        "last_login_days_ago": "mean"

    })
    .reset_index()

)


customer_behaviour_summary.head()

,customer_id,monthly_login_count,average_session_time_minutes,features_used,last_login_days_ago
0,10001,19.666667,28.124427,5.083333,29.666667
1,10002,14.583333,24.804342,5.833333,34.166667
2,10003,14.333333,24.192618,4.916667,30.250000
3,10004,17.083333,22.674429,5.416667,26.583333
4,10005,10.333333,24.052414,4.500000,28.083333


In [27]:
customer_data = (

    customers

    .merge(
        customer_behaviour_summary,
        on="customer_id",
        how="left"
    )

    .merge(
        support_summary,
        on="customer_id",
        how="left"
    )

)


customer_data.shape

(10000, 15)

In [28]:
print("customers:", "customers" in globals())
print("customer_behaviour_summary:", "customer_behaviour_summary" in globals())
print("support_summary:", "support_summary" in globals())

customers: True
customer_behaviour_summary: True
support_summary: True


In [29]:
customer_data.shape

(10000, 15)

In [30]:
customer_data.columns

Index(['customer_id', 'signup_date', 'region', 'age_group',
       'subscription_type', 'monthly_revenue', 'tenure_months',
       'monthly_login_count', 'average_session_time_minutes', 'features_used',
       'last_login_days_ago', 'complaint_count', 'response_time_hours',
       'resolution_time_hours', 'customer_satisfaction_score'],
      dtype='object')

In [31]:
customer_data.isna().sum()

customer_id                        0
signup_date                        0
region                             0
age_group                          0
subscription_type                  0
monthly_revenue                    0
tenure_months                      0
monthly_login_count                0
average_session_time_minutes       0
features_used                      0
last_login_days_ago                0
complaint_count                 2283
response_time_hours             2283
resolution_time_hours           2283
customer_satisfaction_score     2283
dtype: int64

In [32]:
customer_data.fillna(
    {
        "complaint_count": 0,
        "response_time_hours": 0,
        "resolution_time_hours": 0,
        "customer_satisfaction_score": 10
    },
    inplace=True
)

In [33]:
customer_data.isna().sum()

customer_id                     0
signup_date                     0
region                          0
age_group                       0
subscription_type               0
monthly_revenue                 0
tenure_months                   0
monthly_login_count             0
average_session_time_minutes    0
features_used                   0
last_login_days_ago             0
complaint_count                 0
response_time_hours             0
resolution_time_hours           0
customer_satisfaction_score     0
dtype: int64

In [34]:
customer_data.describe()

,customer_id,signup_date,monthly_revenue,tenure_months,monthly_login_count,average_session_time_minutes,features_used,last_login_days_ago,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score
count,10000.00000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,15000.50000,2022-07-07 01:39:38.880000,73.300000,41.946800,14.984033,25.079703,5.003808,29.432383,2.971400,18.961384,47.272140,6.535279
min,10001.00000,2020-01-01 00:00:00,29.000000,12.000000,5.250000,15.615036,2.333333,12.000000,0.000000,0.000000,0.000000,1.000000
25%,12500.75000,2021-04-15 00:00:00,29.000000,27.000000,12.916667,23.141626,4.500000,26.000000,1.000000,3.817181,9.361189,4.333333
50%,15000.50000,2022-07-10 12:00:00,79.000000,42.000000,14.916667,25.042053,5.000000,29.416667,3.000000,20.376485,49.942020,6.500000
75%,17500.25000,2023-09-30 00:00:00,79.000000,57.000000,16.916667,27.006707,5.500000,32.916667,5.000000,29.813771,74.435145,10.000000
max,20000.00000,2025-01-01 00:00:00,199.000000,73.000000,25.666667,35.450841,7.583333,46.833333,15.000000,47.997467,119.988796,10.000000
std,2886.89568,NaN,56.212329,17.461468,2.919387,2.841681,0.744646,5.048072,2.630641,14.173765,35.283320,2.797491


In [35]:
# Introduce realistic disengaged SaaS customers

low_engagement = np.random.choice(
    customer_data.index,
    size=1000,
    replace=False
)


customer_data.loc[
    low_engagement,
    "monthly_login_count"
] = np.random.randint(
    0,
    5,
    size=1000
)


customer_data.loc[
    low_engagement,
    "features_used"
] = np.random.randint(
    1,
    3,
    size=1000
)


customer_data.loc[
    low_engagement,
    "last_login_days_ago"
] = np.random.randint(
    45,
    120,
    size=1000
)

In [36]:
customer_data[
[
"monthly_login_count",
"features_used",
"last_login_days_ago"
]
].describe()

,monthly_login_count,features_used,last_login_days_ago
count,10000.000000,10000.000000,10000.000000
mean,13.688150,4.655050,34.720025
std,4.814294,1.277500,17.964078
min,0.000000,1.000000,12.000000
25%,12.166667,4.250000,26.416667
50%,14.500000,4.916667,30.166667
75%,16.666667,5.416667,34.333333
max,25.666667,7.583333,119.000000


In [37]:
customer_data["churn_score"] = 0

In [38]:
# Engagement risk

customer_data["churn_score"] += np.where(
    customer_data["monthly_login_count"] < 5,
    3,
    0
)


customer_data["churn_score"] += np.where(
    customer_data["features_used"] <= 2,
    2,
    0
)


customer_data["churn_score"] += np.where(
    customer_data["last_login_days_ago"] > 45,
    3,
    0
)

In [39]:
# SUpport Risk

customer_data["churn_score"] += np.where(
    customer_data["complaint_count"] > 5,
    2,
    0
)


customer_data["churn_score"] += np.where(
    customer_data["customer_satisfaction_score"] < 5,
    3,
    0
)

In [40]:
# Subscription risk

customer_data["churn_score"] += np.where(
    customer_data["subscription_type"] == "Basic",
    1,
    0
)

In [41]:
# Convert score into probability

customer_data["churn_probability"] = (
    customer_data["churn_score"] / 12
).clip(0,1)


In [42]:
# Churn Label

customer_data["churn"] = np.where(
    customer_data["churn_probability"] >= 0.4,
    1,
    0
)


Check churn balance

In [43]:
customer_data["churn"].value_counts()

churn
0    8496
1    1504
Name: count, dtype: int64

In [44]:
customer_data["churn"].value_counts(normalize=True) * 100

churn
0    84.96
1    15.04
Name: proportion, dtype: float64

Validate the churn logic

In [45]:
customer_data.groupby("churn")[
[
"monthly_login_count",
"features_used",
"last_login_days_ago",
"complaint_count",
"customer_satisfaction_score"
]
].mean()

,monthly_login_count,features_used,last_login_days_ago,complaint_count,customer_satisfaction_score
churn,,,,,
0,14.978372,5.006905,29.394931,2.706685,6.708328
1,6.399767,2.667442,64.801141,4.466755,5.557738


In [46]:
# Clean the Final Dataset
# Remove the temporary columns

customer_data.drop(
    columns=[
        "churn_score",
        "churn_probability"
    ],
    inplace=True
)

In [47]:
customer_data.columns

Index(['customer_id', 'signup_date', 'region', 'age_group',
       'subscription_type', 'monthly_revenue', 'tenure_months',
       'monthly_login_count', 'average_session_time_minutes', 'features_used',
       'last_login_days_ago', 'complaint_count', 'response_time_hours',
       'resolution_time_hours', 'customer_satisfaction_score', 'churn'],
      dtype='object')

In [48]:
# Check final data set

customer_data.shape

(10000, 16)

In [49]:
customer_data["churn"].value_counts()

churn
0    8496
1    1504
Name: count, dtype: int64

In [50]:
# Save the final analytical dataset

customer_data.to_csv(
    "customer_churn_final.csv",
    index=False
)

In [51]:
# Usuable customers table 

customers_final = customer_data[
[
"customer_id",
"signup_date",
"region",
"age_group",
"subscription_type",
"monthly_revenue",
"tenure_months",
"churn"
]
]

**Export SQLite database**

In [52]:
import sqlite3

conn = sqlite3.connect(
    "customer_churn_analytics.db"
)


customers_final.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)


customer_behaviour.to_sql(
    "customer_behaviour",
    conn,
    if_exists="replace",
    index=False
)


support_interactions.to_sql(
    "support_interactions",
    conn,
    if_exists="replace",
    index=False
)


conn.close()

In [53]:
import os

os.listdir()

['__notebook__.ipynb',
 'customer_churn_analytics.db',
 'customer_churn_final.csv']

In [54]:
import sqlite3
import pandas as pd


conn = sqlite3.connect(
    "customer_churn_analytics.db"
)


tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)


tables

,name
0,customers
1,customer_behaviour
2,support_interactions
